# CENG 476 — Plant Disease Classification Using Deep Learning and Transfer Learning

**Student:** Emir EVREN — **ID:** 210444038  
**Framework:** PyTorch

This notebook is organized around the technical requirements of the CENG 476 project specification. It documents the implementation, parameter values, recorded outputs, experimental comparisons, and the reason for each major design decision. Expensive GPU training is not automatically repeated when the notebook is opened; recorded outputs from the completed runs are retained for inspection.

**Final audited benchmark:** Custom CNN **84.62%**, ResNet18 **97.66%**, EfficientNet-B0 **99.01%**, 50/50 ensemble **99.14%**.

## 1. Requirement Coverage

| Assignment item | Implementation |
|---|---|
| Custom architecture | Four-block CNN trained from scratch |
| Transfer learning | ImageNet-pretrained ResNet18 and EfficientNet-B0 |
| Batch Normalization | After every convolution in the custom CNN |
| Dropout | 0.40 baseline; 0.30 transfer classifiers |
| Regularization | Data augmentation + AdamW weight decay `1e-4` |
| Over/underfitting analysis | Augmented train + clean train + validation + test |
| Optimizer | AdamW, betas `(0.9,0.999)` |
| LR tuning | baseline pilots `1e-3`, `5e-4`, `3e-4` |
| Scheduler | ReduceLROnPlateau |
| Early stopping | patience 6 baseline, 5 transfer |
| Activations | ReLU; EfficientNet native SiLU |
| Evaluation | train / validation / locked test; no k-fold CV |
| Metrics | accuracy, precision, recall, Macro/Weighted F1, ROC-AUC, confusion matrix |
| Creative experiment | validation-selected soft-voting ensemble |
| Reproducibility | seed 42 + seeds 123/777 stability check |

## 2. Task, Dataset and Three-Way Split

The task is **single-label 38-class image classification**. Input tensors have shape `[B,3,224,224]`; the classifier returns `[B,38]` raw logits.

A fixed **train / validation / test** protocol is used. Training data supplies gradients; validation data supports scheduler, checkpoint, hyperparameter and ensemble-weight decisions; the locked test is reserved for final evaluation.

**K-fold cross-validation was not used.** Repeated full CNN fine-tuning across folds would substantially increase compute cost. A dedicated validation split plus locked test was already maintained, and stability was separately evaluated with multiple random seeds.

The test set was not used for model training, checkpoint selection, hyperparameter tuning or ensemble-weight selection. Test images participated only in deterministic, model-independent integrity auditing.

In [1]:
initial={'train':43444,'validation':5430,'test':5431}
final={'train':39091,'validation':4462,'test':10709}

print('INITIAL IMAGE-LEVEL PROTOCOL')
for k,v in initial.items(): print(f'{k:10s}: {v:,}')
print('total     :',f'{sum(initial.values()):,}')

print('\nFINAL ULTRA-STRICT PROTOCOL')
for k,v in final.items(): print(f'{k:10s}: {v:,}')
print('total     :',f'{sum(final.values()):,}')
print('classes   : 38')

INITIAL IMAGE-LEVEL PROTOCOL
train     : 43,444
validation: 5,430
test      : 5,431
total     : 54,305

FINAL ULTRA-STRICT PROTOCOL
train     : 39,091
validation: 4,462
test      : 10,709
total     : 54,262
classes   : 38


## 3. Reproducibility

The reference random seed is **42**. Python, NumPy and PyTorch are seeded; on CUDA, all GPU seeds are set and deterministic cuDNN behavior is requested. The seed is not mathematically special; it provides a reproducible reference run.

In [2]:
import random
import numpy as np
import torch

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

print('Reference seed:',SEED)
print('Python / NumPy / PyTorch seeded: yes')

Reference seed: 42
Python / NumPy / PyTorch seeded: yes


## 4. Preprocessing and Data Augmentation

**Training only:** `RandomResizedCrop(224, scale=(0.80,1.00))`, horizontal flip with probability 0.5, rotation ±15°, ColorJitter with brightness/contrast/saturation 0.2, tensor conversion, then ImageNet normalization.

**Validation/test:** `Resize(256) -> CenterCrop(224) -> ToTensor() -> Normalize`.

Augmentation is limited to training because random evaluation transforms would add measurement noise. The augmentation is intentionally moderate so disease-related color and texture are not deliberately destroyed. ImageNet normalization is used because the transfer models were pretrained with these channel statistics.

In [3]:
from torchvision import transforms
from torchvision.transforms import InterpolationMode

MEAN=[0.485,0.456,0.406]
STD=[0.229,0.224,0.225]

train_transform=transforms.Compose([
    transforms.RandomResizedCrop(224,scale=(0.80,1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15,interpolation=InterpolationMode.BILINEAR,fill=(128,128,128)),
    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])

evaluation_transform=transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])

print(train_transform)
print()
print(evaluation_transform)

Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0))
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0])
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2), hue=None)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Compose(
    Resize(size=256)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## 5. DataLoader and Batch Dimensions

The baseline uses batch size **64**; transfer models use **32**. The deterministic clean-train evaluation subset contains **20 images per class = 760 images**. It is not used to update model parameters; it exists to compare training and validation behavior without random augmentation.

In [4]:
print('Baseline image batch : torch.Size([64, 3, 224, 224])')
print('Baseline label batch : torch.Size([64])')
print('Transfer batch size  :',32)
print('Clean train eval     :',38*20)

Baseline image batch : torch.Size([64, 3, 224, 224])
Baseline label batch : torch.Size([64])
Transfer batch size  : 32
Clean train eval     : 760


## 6. Custom CNN Architecture

Four repeated blocks are used: `Conv(3x3) -> BatchNorm -> ReLU -> MaxPool(2x2)` with channels `32 -> 64 -> 128 -> 256`. The feature extractor is followed by `AdaptiveAvgPool(1x1) -> Flatten -> Dropout(0.40) -> Linear(256,38)`.

Convolution is appropriate because disease symptoms are spatially local visual structures such as lesions, texture and discoloration. `padding=1` preserves height/width through each 3x3 convolution; max pooling halves spatial dimensions.

Spatial path: `3x224x224 -> 32x112x112 -> 64x56x56 -> 128x28x28 -> 256x14x14 -> 256x1x1 -> 38 logits`.

In [5]:
from torch import nn

class BaselineCNN(nn.Module):
    def __init__(self,num_classes=38,dropout_rate=0.4):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(inplace=True),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(inplace=True),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(inplace=True),nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU(inplace=True),nn.MaxPool2d(2),
        )
        self.pool=nn.AdaptiveAvgPool2d((1,1))
        self.classifier=nn.Sequential(nn.Flatten(),nn.Dropout(dropout_rate),nn.Linear(256,num_classes))
    def forward(self,x):
        return self.classifier(self.pool(self.features(x)))

model=BaselineCNN()
x=torch.randn(2,3,224,224)
with torch.inference_mode():
    y=model.eval()(x)
print('Input :',tuple(x.shape))
print('Output:',tuple(y.shape))
print('Trainable parameters:',f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Input : (2, 3, 224, 224)
Output: (2, 38)
Trainable parameters: 399,142


## 7. Batch Normalization, Dropout and Regularization

**Batch Normalization** is placed after every convolution and before ReLU. It stabilizes intermediate activation scales and makes optimization more reliable. During training it uses mini-batch statistics; during evaluation it uses stored running statistics.

**Dropout** is active only during training. Final values are **0.40** for the custom CNN and **0.30** in the transfer-model classifiers.

Other explicit regularization methods are training data augmentation and AdamW weight decay `1e-4`. Early stopping is implemented as an additional safeguard.

In [6]:
bn=[m for m in model.modules() if isinstance(m,nn.BatchNorm2d)]
print('BatchNorm layers:',len(bn))
print('Order: Conv -> BatchNorm -> ReLU -> MaxPool')
print()
for d,f1 in [(0.20,0.4990),(0.40,0.4953),(0.60,0.4311)]:
    print(f'dropout={d:.2f}  best validation Macro-F1={f1:.4f}')

BatchNorm layers: 4
Order: Conv -> BatchNorm -> ReLU -> MaxPool

dropout=0.20  best validation Macro-F1=0.4990
dropout=0.40  best validation Macro-F1=0.4953
dropout=0.60  best validation Macro-F1=0.4311


The dropout pilot shows that **0.60 was too aggressive** and slowed learning. The 0.20/0.40 difference was small. The predefined final baseline remained at 0.40; test performance was not used to retroactively select dropout.

## 8. Activation Functions, Logits, Softmax and Loss

The custom CNN and ResNet18 use **ReLU**. EfficientNet-B0 retains its native **SiLU** activation. Nonlinear activation is necessary because stacking only linear transformations remains equivalent to another linear transformation.

The output layer returns **38 raw logits**. No Softmax is placed before the loss. The project uses `nn.CrossEntropyLoss()`, which expects raw logits and internally performs the stable LogSoftmax/NLL computation. Softmax is used later only when probabilities are needed for ensemble voting, confidence and calibration.

In [7]:
criterion=nn.CrossEntropyLoss()
logits=torch.tensor([[2.0,0.5,-1.0],[0.1,1.8,0.2]])
labels=torch.tensor([0,1])
loss=criterion(logits,labels)
probs=torch.softmax(logits,dim=1)
print('loss:',f'{loss.item():.4f}')
print('first probability vector:',[round(v,4) for v in probs[0].tolist()])
print('sum:',f'{probs[0].sum().item():.4f}')

loss: 0.3161
first probability vector: [0.7856, 0.1753, 0.0391]
sum: 1.0000


## 9. Optimizer, Learning Rate, Scheduler and Early Stopping

All final models use **AdamW**, betas `(0.9,0.999)`, weight decay `1e-4`. Baseline LR pilots included `1e-3`, `5e-4`, `3e-4`; the final baseline uses `5e-4`.

Transfer models use differential learning rates: pretrained backbone `1e-4`, new classifier `5e-4`. The backbone is updated more conservatively because it already contains useful ImageNet features.

`ReduceLROnPlateau` monitors validation loss with factor **0.5**, patience **2**, minimum LR `1e-6`. Validation loss is a smooth scheduler signal, while **validation Macro-F1** is used for checkpoint selection because it balances the 38 classes. On an F1 tie, lower validation loss is preferred.

Early-stop patience: **6 baseline**, **5 transfer**. The selected final runs reached their configured maximum epoch before early stopping terminated training.

In [8]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
p0=nn.Parameter(torch.zeros(1))
opt=AdamW([p0],lr=5e-4,betas=(0.9,0.999),weight_decay=1e-4)
scheduler=ReduceLROnPlateau(opt,mode='min',factor=0.5,patience=2,min_lr=1e-6)
print('optimizer: AdamW')
print('betas:',opt.defaults['betas'])
print('weight_decay:',opt.defaults['weight_decay'])
print('scheduler: ReduceLROnPlateau')
print('factor=0.5 patience=2 min_lr=1e-6')
print('checkpoint metric: validation Macro-F1')
print('baseline patience: 6')
print('transfer patience: 5')

optimizer: AdamW
betas: (0.9, 0.999)
weight_decay: 0.0001
scheduler: ReduceLROnPlateau
factor=0.5 patience=2 min_lr=1e-6
checkpoint metric: validation Macro-F1
baseline patience: 6
transfer patience: 5


### Final hyperparameters

| Setting | Custom CNN | Transfer models |
|---|---:|---:|
| max epochs | 15 | 12 |
| batch size | 64 | 32 |
| LR/backbone LR | `5e-4` | `1e-4` |
| classifier LR | — | `5e-4` |
| dropout | 0.40 | 0.30 |
| weight decay | `1e-4` | `1e-4` |
| early-stop patience | 6 | 5 |

## 10. Training Loop and AMP

Mini-batch order: clear old gradients -> forward pass -> CrossEntropyLoss -> backward -> optimizer step -> GradScaler update. PyTorch accumulates gradients by default, therefore `zero_grad` is required. AMP is enabled on CUDA to reduce memory use and accelerate supported operations.

In [9]:
def train_one_batch(model,images,labels,criterion,optimizer,scaler,device):
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=(device.type=='cuda')):
        logits=model(images)
        loss=criterion(logits,labels)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    return loss,logits.argmax(1)

print('1 zero_grad')
print('2 forward')
print('3 CrossEntropyLoss')
print('4 backward')
print('5 optimizer.step')
print('6 scaler.update')

1 zero_grad
2 forward
3 CrossEntropyLoss
4 backward
5 optimizer.step
6 scaler.update


## 11. Sanity Check Before Full Training

Before long runs, the custom CNN was deliberately overfit on **38 images** with augmentation disabled, dropout 0 and weight decay 0. This is an implementation check, not a generalization experiment.

In [10]:
print('Step 30: accuracy = 71.05%')
print('Step 40: accuracy = 100.00%')
print('Sanity check: PASS')

Step 30: accuracy = 71.05%
Step 40: accuracy = 100.00%
Sanity check: PASS


Reaching 100% indicates that the model, labels, loss, backpropagation and optimizer can fit a tiny dataset. Failure would suggest a pipeline bug or insufficient optimization.

## 12. Transfer Learning Models

**ResNet18:** ImageNet pretrained, full fine-tuning; classifier `Dropout(0.30) -> Linear(512,38)`. Residual connections support gradient flow.

**EfficientNet-B0:** ImageNet pretrained, full fine-tuning; classifier `Dropout(0.30) -> Linear(1280,38)`. It provides the strongest individual performance/parameter trade-off.

In [11]:
for name,n in [('Custom CNN',399142),('ResNet18',11196006),('EfficientNet-B0',4056226)]:
    print(f'{name:16s}: {n:,}')

Custom CNN      : 399,142
ResNet18        : 11,196,006
EfficientNet-B0 : 4,056,226


## 13. Metrics and Final Results

Accuracy measures total correctness. Precision = TP/(TP+FP), Recall = TP/(TP+FN), and F1 is their harmonic mean. **Macro-F1** averages the 38 class-wise F1 scores equally and is therefore preferred for checkpoint selection when class support differs. Weighted-F1 weights by class support. Confusion matrices show which classes are confused. Multiclass ROC-AUC is evaluated one-vs-rest.

In [12]:
import pandas as pd
results=pd.DataFrame([
 ['Custom CNN',84.620413,0.8635,0.7666,0.781323,0.836106,1647],
 ['ResNet18',97.656177,0.9734,0.9676,0.968603,0.976314,251],
 ['EfficientNet-B0',99.010178,0.9897,0.9855,0.987373,0.990114,106],
 ['50/50 Ensemble',99.140910,0.9908,0.9891,0.989733,0.991396,92]],
 columns=['model','accuracy_%','macro_precision','macro_recall','macro_f1','weighted_f1','errors'])
print(results.to_string(index=False))
print()
for x in [('Custom CNN',95.48,0.995589),('ResNet18',99.79,0.999929),('EfficientNet-B0',99.91,0.999955),('Ensemble',99.95,0.999976)]:
    print(f'{x[0]:16s} top3={x[1]:.2f}% AUC={x[2]:.6f}')

           model  accuracy_%  macro_precision  macro_recall  macro_f1  weighted_f1  errors
      Custom CNN   84.620413           0.8635        0.7666  0.781323     0.836106    1647
        ResNet18   97.656177           0.9734        0.9676  0.968603     0.976314     251
 EfficientNet-B0   99.010178           0.9897        0.9855  0.987373     0.990114     106
  50/50 Ensemble   99.140910           0.9908        0.9891  0.989733     0.991396      92

Custom CNN       top3=95.48% AUC=0.995589
ResNet18         top3=99.79% AUC=0.999929
EfficientNet-B0  top3=99.91% AUC=0.999955
Ensemble         top3=99.95% AUC=0.999976


## 14. Validation-Selected Soft-Voting Ensemble

Soft voting combines probability distributions rather than hard class labels: `p = w_R*p_ResNet + w_E*p_EfficientNet`. Candidate weights were selected **only on validation data**.

In [13]:
cand=pd.DataFrame([
 ['ResNet only',1,0,.988122,.986237],
 ['75R/25E',.75,.25,.990587,.988722],
 ['50R/50E',.5,.5,.993725,.992033],
 ['25R/75E',.25,.75,.989691,.986400],
 ['Eff only',0,1,.986777,.982431]],
 columns=['candidate','wR','wE','val_acc','val_macro_f1'])
print(cand.to_string(index=False))
print('\nSelected: 50R/50E')

  candidate   wR   wE  val_acc  val_macro_f1
ResNet only 1.00 0.00 0.988122      0.986237
    75R/25E 0.75 0.25 0.990587      0.988722
    50R/50E 0.50 0.50 0.993725      0.992033
    25R/75E 0.25 0.75 0.989691      0.986400
    Eff only 0.00 1.00 0.986777      0.982431

Selected: 50R/50E


The 50/50 candidate had the highest validation Macro-F1 (**0.992033**) and was fixed before test evaluation.

## 15. Overfitting / Underfitting Analysis

A deterministic balanced clean-train subset is compared with validation and test. Augmented training metrics should not be treated as directly equivalent to deterministic evaluation metrics.

In [14]:
g=pd.DataFrame([
 ['Custom CNN',95.3979,77.8947,83.55,84.6204],
 ['ResNet18',99.6905,98.1579,98.8122,97.6562],
 ['EfficientNet-B0',99.4167,99.7368,98.6777,99.0102]],
 columns=['model','aug_train_%','clean_train_%','val_%','test_%'])
print(g.to_string(index=False))

           model  aug_train_%  clean_train_%   val_%  test_%
      Custom CNN      95.3979        77.8947 83.5500 84.6204
        ResNet18      99.6905        98.1579 98.8122 97.6562
 EfficientNet-B0      99.4167        99.7368 98.6777 99.0102


EfficientNet clean-train, validation and PlantVillage test scores remain close, so severe **classical train-set overfitting** is not supported. This does not imply strong cross-domain generalization.

## 16. Leakage Audit and Protocol Revision

The historical image-level ensemble reached **99.76%**, which motivated an integrity audit. File-level disjointness was insufficient because different image files may correspond to the same physical leaf. The audit checked exact hashes, perceptual near-duplicates and mapped physical-leaf identity.

In [15]:
audit=pd.DataFrame([
 ['Exact cross-split duplicates',10,0],
 ['Perceptual near-duplicate pairs',68,0],
 ['Mapped same-leaf cross-split groups',4956,0],
 ['Strict dHash<=4 pairs',39,0]],columns=['check','before','after_ultra'])
print(audit.to_string(index=False))
print('\nQuarantine: train=34, validation=4, test=0')
print('Priority: test > validation > train')

                               check  before  after_ultra
        Exact cross-split duplicates      10            0
       Perceptual near-duplicate pairs      68            0
Mapped same-leaf cross-split groups    4956            0
              Strict dHash<=4 pairs      39            0

Quarantine: train=34, validation=4, test=0
Priority: test > validation > train


`dHash` is a perceptual image hash. Hamming distance `<=4` was used as the strict near-duplicate criterion. Physical-leaf mapping covers approximately **75.7%** of images; therefore the project does not claim mathematical proof of uniqueness for every unmapped specimen. The defensible statement is that no **detected** exact, mapped-leaf or strict dHash<=4 cross-split overlap remains under the audit protocol.

## 17. Historical vs Final Results

The original protocol was optimistic. The complete accuracy difference is **not** attributed solely to leakage because the protocol and training conditions also changed.

In [16]:
h=pd.DataFrame([
 ['Custom CNN',87.42,84.620413],
 ['ResNet18',99.26,97.656177],
 ['EfficientNet-B0',99.52,99.010178],
 ['Ensemble',99.76,99.140910]],columns=['model','historical_%','ultra_%'])
h['delta_pp']=h['ultra_%']-h['historical_%']
print(h.to_string(index=False))

           model  historical_%   ultra_%  delta_pp
      Custom CNN         87.42 84.620413 -2.799587
        ResNet18         99.26 97.656177 -1.603823
 EfficientNet-B0         99.52 99.010178 -0.509822
        Ensemble         99.76 99.140910 -0.619090


## 18. Seed Stability and Random-Label Control

Three EfficientNet-B0 runs on the same final manifest test whether the high result depends on a lucky seed. A separate random-label negative control checks for trivial pipeline/label leakage.

In [17]:
seeds=pd.DataFrame([[42,99.010178,.987373,106],[123,98.767392,.983668,132],[777,99.224951,.990149,83]],columns=['seed','acc_%','macro_f1','errors'])
print(seeds.to_string(index=False))
print('\nmean accuracy=99.001%')
print('std=0.229 percentage points')
print('range=0.458 percentage points')
print('\n38-class chance accuracy : 2.63%')
print('true-label val accuracy  : 1.8421%')
print('val Macro-F1             : 0.01832')
print('random-label result      : PASS')

 seed     acc_%  macro_f1  errors
   42 99.010178  0.987373     106
  123 98.767392  0.983668     132
  777 99.224951  0.990149      83

mean accuracy=99.001%
std=0.229 percentage points
range=0.458 percentage points

38-class chance accuracy : 2.63%
true-label val accuracy  : 1.8421%
val Macro-F1             : 0.01832
random-label result      : PASS


Seed 777 is not promoted as the main benchmark because selecting the highest test seed after seeing the outcomes would be cherry-picking. The random-label result is evidence against trivial leakage, not proof that every possible form of leakage is absent.

## 19. Calibration, Bootstrap and Robustness

ECE measures the gap between confidence and empirical correctness. NLL and Brier score evaluate probabilistic predictions. Confidence intervals use **1,000 ordinary bootstrap resamples**. Robustness experiments are post-hoc diagnostics; the locked test was not used to retune the models.

In [18]:
cal=pd.DataFrame([['EfficientNet',.003845,.035704,.016145],['Ensemble',.009121,.035419,.017435]],columns=['model','ECE','NLL','Brier'])
print(cal.to_string(index=False))
print('\nEfficientNet accuracy 95% CI approx: 98.81–99.20%')
print('Ensemble accuracy 95% CI approx    : 98.95–99.31%')

r=pd.DataFrame([
 ['clean',99.0102,99.1409],['brightness .60',98.8328,99.1783],['brightness 1.40',98.0670,98.7300],
 ['contrast .60',98.4873,98.7861],['Gaussian blur r2',84.0788,86.5254],['JPEG30',98.1324,98.7207],
 ['rotation15',99.4117,99.5051],['center occlusion60',55.3833,64.4505],['border keep60',58.8477,77.9344]],
 columns=['condition','Eff_%','Ens_%'])
print('\n'+r.to_string(index=False))

       model      ECE      NLL    Brier
EfficientNet 0.003845 0.035704 0.016145
    Ensemble 0.009121 0.035419 0.017435

EfficientNet accuracy 95% CI approx: 98.81–99.20%
Ensemble accuracy 95% CI approx    : 98.95–99.31%

          condition   Eff_%   Ens_%
              clean 99.0102 99.1409
     brightness .60 98.8328 99.1783
    brightness 1.40 98.0670 98.7300
       contrast .60 98.4873 98.7861
   Gaussian blur r2 84.0788 86.5254
             JPEG30 98.1324 98.7207
         rotation15 99.4117 99.5051
center occlusion60 55.3833 64.4505
      border keep60 58.8477 77.9344


Brightness, contrast, JPEG compression and rotation are tolerated relatively well; blur and large occlusions produce much larger degradation. These stress tests suggest sensitivity to fine visual detail but do not prove a causal shortcut-learning mechanism. Grad-CAM was also used as qualitative supportive evidence, not causal proof.

## 20. PlantDoc External OOD Evaluation

The final models were tested **without retraining** on 236 mapped Cropped-PlantDoc test images across 27 mapped source classes. Mapping between PlantDoc and PlantVillage labels is manual, so this is an out-of-distribution stress test rather than a directly comparable benchmark.

In [19]:
plantdoc=pd.DataFrame([['EfficientNet-B0',23.31,.2183,181],['50/50 Ensemble',25.00,.2349,177]],columns=['model','acc_%','mapped_macro_f1','errors'])
print(plantdoc.to_string(index=False))

           model  acc_%  mapped_macro_f1  errors
 EfficientNet-B0  23.31           0.2183     181
  50/50 Ensemble  25.00           0.2349     177


The PlantDoc drop does not by itself prove image memorization because PlantVillage clean-train, validation and test performance are all high. The stronger interpretation is **strong within-domain generalization but weak cross-domain transfer / dataset dependence**. Near-99% PlantVillage accuracy must therefore not be interpreted as near-99% real-world field accuracy.

## 21. Experimental Development Summary

| Change | Reason | Outcome |
|---|---|---|
| 38-image sanity test | validate pipeline | 100%, PASS |
| Custom CNN | scratch baseline | 84.62% |
| LR pilots | improve convergence | baseline 5e-4 |
| Dropout pilot | regularization study | 0.60 too aggressive |
| ResNet18 transfer | evaluate pretrained features | 97.66% |
| EfficientNet-B0 | parameter-efficient transfer model | 99.01% |
| Differential LR | preserve pretrained backbone | 1e-4 / 5e-4 |
| Scheduler | handle validation plateau | factor .5, patience 2 |
| Ensemble | combine model probabilities | 99.14%, 92 errors |
| Leakage audit | investigate 99.76% | overlap detected |
| Ultra-strict protocol | stronger evaluation | 99.14% final |
| Multiple seeds | reproducibility | 99.001 +/- 0.229 pp |
| Random labels | leakage sanity check | chance-level, PASS |
| Calibration/bootstrap | probabilistic uncertainty | low ECE, narrow CI |
| PlantDoc OOD | external validity | 23–25% |

## 22. Conclusion and Reproduction

The scratch CNN provides a transparent baseline. Transfer learning produces a large improvement. EfficientNet-B0 is the strongest individual model (**99.01%**); the validation-selected 50/50 ensemble gives the strongest same-domain prediction (**99.14%**, Macro-F1 **0.9897**).

The historical image-level protocol was optimistic and contained documented overlap. High PlantVillage performance nevertheless survives the stricter protocol and is stable across seeds. The main limitation is cross-domain generalization: PlantDoc accuracy is only 23–25%.

**Audit limitation:** no detected exact, mapped-leaf or dHash<=4 overlap remains under the implemented audit, but leaf-identity mapping does not cover every image.

Reproduction commands:
```bash
python src/train_baseline.py --epochs 15 --learning-rate 5e-4 --batch-size 64 --weight-decay 1e-4 --dropout 0.4
python src/train_resnet18_official.py
python src/train_efficientnet_official.py
python src/full_control_core.py
python src/evaluate_plantdoc_ood.py
```

Heavy training is not automatically repeated when the notebook opens. The recorded outputs above remain visible, while full execution logic is maintained in `src/`.